# Introduction

|Term| Definitions | Example/Explanation | 
|-----------|--------|-------|
|Time series | A collection of random variables indexed according to the order they are obtained in time. | Consider a sequence of random variables $x_1, x_2, x_3, .. $ <br> where $x_1$ denotes the random variable where the values taken by the seires at the first time point etc|
| Stochastic process | Collection of random vairables $\{x_t\}$, indexed by $t$ | The values $t \in \mathbb{Z}$ and the observed values is reffered as $realization$ of the Stochastic process|

The graphs that are plotted will usually contain discrete and finite sampled values at regular intervals, even though they could be continously sampled.<br> The main reason for this is computational limitation. If the data isn't sampled at a fast enough rate then the graph can be completely different to the true structure and thus mislead to inaccurate conclusions, this is known at $aliasing$ 

## White Noise and its motivation

|Term| Definitions | Example/Explanation | 
|-----------|--------|-------|
| White Noise | A collection of uncorrelated random variables $w_t$, with $\mu = 0$ and finite variance $\sigma_w^2$ | - Standard white noise $w_t$ ~ $wn(0, \sigma_w^2)$ <br> - $iid$ noise $w_t$ ~ $iid(0, \sigma_w^2)$ <br> - Guassian White noise $w_t$ ~ $iid N(0, \sigma_w^2)$|

We start with white noise since it's base line where all sequential complexity roots from.<br>
We will soon see that different models are actually filters applied to white noise. 
In time series analysis, the whole challenge is that data points are correlated over time.<br>
Introducing white noise first establishes the null hypothesis: a scenario where the past tells you absolutely nothing about the future.<br>
You have to understand the baseline of "no pattern" before you can start modeling complex patterns.

### Time Series Goal: 
When you fit an algorithm to forecast a time series, your goal is to extract every ounce of signal out of the data capturing the trend, the seasonality, and the autocorrelation.<br>
How do you know if your model is actually good? You look at the leftovers (the residuals).<br> If your model successfully captured all the patterns, the residuals should look exactly like pure, unpredictable white noise. If your residuals still show a pattern, your model left information on the table.

In [4]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set seed for reproducibility
np.random.seed(42)
n_steps = 200

# Step 1: The Foundation - Gaussian White Noise
white_noise = np.random.normal(0, 1, n_steps)

# Step 2: Introducing Dependency - Random Walk
random_walk = np.zeros(n_steps)
for t in range(1, n_steps):
    random_walk[t] = random_walk[t-1] + white_noise[t]

# Step 3: Adding Autocorrelation - Autoregressive (AR) Process
ar_process = np.zeros(n_steps)
phi = 0.8
for t in range(1, n_steps):
    ar_process[t] = phi * ar_process[t-1] + white_noise[t]

# Step 4: A Complex Series - AR Process + Trend + Seasonality
time = np.arange(n_steps)
trend = 0.05 * time
seasonality = 5 * np.sin(2 * np.pi * time / 50)
complex_series = ar_process + trend + seasonality

# --- Plotting the Progression using Plotly ---
fig = make_subplots(
    rows=4, cols=1, 
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        '1. The Baseline: Gaussian White Noise (Unpredictable, Independent)',
        '2. Cumulative Dependency: Random Walk',
        '3. Partial Dependency: AR(1) Process (phi = 0.8)',
        '4. Complex Real-World Series (AR + Trend + Seasonality)'
    )
)

# Add traces to their respective subplots
fig.add_trace(go.Scatter(y=white_noise, mode='lines', name='White Noise', line=dict(color='gray')), row=1, col=1)
fig.add_trace(go.Scatter(y=random_walk, mode='lines', name='Random Walk', line=dict(color='blue')), row=2, col=1)
fig.add_trace(go.Scatter(y=ar_process, mode='lines', name='AR(1)', line=dict(color='purple')), row=3, col=1)
fig.add_trace(go.Scatter(y=complex_series, mode='lines', name='Complex Series', line=dict(color='red')), row=4, col=1)

# Update layout for a cleaner interactive experience
fig.update_layout(
    height=900, 
    showlegend=False,
    title_text="Building Complexity: From White Noise to Real-World Time Series",
    hovermode="x unified" # Shows values for all plots simultaneously on hover
)

# Show the interactive plot
fig.show()

|Term| Definitions | Example/Explanation | 
|-----------|--------|-------|
|Moving Averages | $$v_t = \frac{1}{3}(w_{t-1} + w_t +w_{t+1})$$ |This is method that smooths the seires,<br> in the definition it took a white noise seires though could be applied anywhere else|
| AutoRegressions ($n^{th}$ order) | A regression model, predicting the current value $x_t$ of a time series as a function of the past $n$ values of the series | - $x_t = ax_{t-1} - bx_{t-2} + cw_t$ <br> - One aspect to consider is how to initialise the first two values|
| Random Walk with Drift | $$x_t = \delta + x_{t-1} + w_t \\ x_t = \delta \cdot t + \sum_{j=1}^t w_j$$ | - Initialise $x_0 = 0$ <br> - $\delta$ is the drift term <br> - $w_t$ is white noise <br> - when $\delta = 0$ this is referred to random walk | 
| Additive Models | $$x_t = s_t + v_t$$ | - $s_t$ unknown signal <br> - $v_t$ a time series may be white or correlated over time <br> - In Additive models, the goal is detect a signal and estimate/extract its waveform

## Measures of Dependence

|Term| Definitions | Example/Explanation | 
|-----------|--------|-------|
|Complete Time series description | $$F(c_1, c_2, \dots, c_n) = Pr(x_{t_1} \le c_1, x_{t_2}\le c_2, \dots, x_{t_n} \le c_n)$$ | This is a joint distribution function <br> - This is (for the most part) purely a mathematical defintion since it's very hard to practically apply it. |
|Marginal Distribution functions | $$F_t(x) = \{x_t \le x\}$$ | - The corresponding marginal density functions are:  $$f_t(x) = \frac{\partial F_t(x)}{\partial x}$$ <br> - When they exists they're very informative on the marginal behviour of the series|

### Mean function 

$$\boxed{\mu_{xt} = \mathbb{E}(x_t) = \int_{-\infty}^{\infty}xf_t(x)}$$

|Applied Case | Working | Conclusion | 
|------------|----------|------------|
|Moving Averages | $$\mu_{vt} = \mathbb{E}(v_t) = \frac{1}{3}\left[\mathbb{E}(w_{t-1})  + \mathbb{E}(w_t) + \mathbb{E}(w_{t+1}) \right] = 0$$ | $$\mu_{vt} = 0 $$| 
|Random Walk with drift | $$\mu_{xt}  = \mathbb{E}(x_t) = \mathbb{E} \left[\delta \cdot t + \sum_{j=1}^t w_j \right] = \mathbb{E} (\delta \cdot t) + \mathbb{E} (\sum_{j=1}^t w_j) = \delta t $$ | $$\mu_{xt} = \delta t$$ |
|Signal Plus Noise | $$\mu_{xt} =  \mathbb{E}(x_t) = \mathbb{E} \left[2\cos(2\pi\frac{t}{50} + 0.6\pi) + w_t\right] = 2\cos(2\pi\frac{t}{50} + 0.6\pi)$$

### Autocovariance function

$$\boxed{\gamma_x(s,t) = \text{cov}(x_s, x_t) = \mathbb{E}\left[(x_s - \mu_s)(x_t - \mu_t)\right]}$$

It's a measure of **linear dependence** between **two time points** in the **same** series.

|Properties | Proof | 
|-----------|-------|
|$$\gamma_x(s,t) = \gamma_x(t, s)$$| $$\text{Let s, t be some time points then } \\ \gamma_x(s,t)= \mathbb{E}\left[(x_s - \mu_s)(x_t - \mu_t)\right] = \mathbb{E}\left[(x_t - \mu_t)(x_s - \mu_s)\right] = \gamma_x(t, s)$$
| $$ \forall s=t \quad \gamma_x(s,t) = \text{var}(x_t)$$ | $$\gamma_x(s,t) = \gamma_x(t,t) = \mathbb{E}\left[(x_t - \mu_t)^2\right] = \text{var}(x_t)$$|

|Applied Case | Working | Conclusion | 
|------------|----------|------------|
|Moving Averages | $$\gamma_w(s,t) = \text{cov}(w_s, w_t)\\ = \mathbb{E}\left[(w_s - \mu_s)(w_t - \mu_t)\right] \\ = \mathbb{E} \left[(w_s - 0)(w_t - 0)\right] = \mathbb{E}(w_sw_t) \\ \text{if } t \ne s \text{ then as they're aren't correlated }\\  \mathbb{E}(w_sw_t) = 0 \\ \mathbb{E}(w_sw_t) = \mathbb{E}(w_t^2) = \sigma_w^2 $$ | $$\gamma_w(s,t) = \begin{cases} \sigma_w^2 & \text{if } s = t \\0 & \text{if } s \neq t\end{cases}$$| 
|Random Walk with drift |
|Signal Plus Noise |